# Milky Way-est: LMC-associated subhalos

**Rockstar and Symfind main-branch selection**

For each Milky Way-est host, we identify the LMC analog using the index mapping from `LMC_idx_indentifier_mwest.ipynb`. The pre-infall snapshot is the snapshot immediately before the first crossing inside the MW host virial radius. A subhalo is associated if it was within one LMC virial radius at any snapshot through that epoch.

Rockstar and Symfind are evaluated independently using their own tracked positions and `ok` flags. In both cases, the boundary is the **Rockstar LMC virial radius** and the infall epoch is shared. The MW central host and LMC analog are excluded. Membership is historical, so no `z=0` survival cut is imposed here.

## 1. Paths and imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import symlib

from lmc_selection import LMC_INDEX_LEGACY, load_catalogs, classify, membership_table

BASE_MWEST = "/data/darkmatter/programs/symlib/standard/"
SUITE = "MWest"
HOST_INDEX = 0
OUTDIR = Path("selection_output")
OUTDIR.mkdir(parents=True, exist_ok=True)

## 2. Selection for one host

`sub_idx` is the symlib main-branch index, not a raw-HLIST `halo_id`.

In [ ]:
c = load_catalogs(BASE_MWEST, SUITE, HOST_INDEX)
i_pre, rock_mask, sym_mask = classify(c)
selected = membership_table(c, i_pre, rock_mask, sym_mask)

print(f"{c.host_name}: LMC index {c.lmc_idx}, pre-infall snapshot {i_pre}, a = {c.scale[i_pre]:.4f}")
print(f"Rockstar: {rock_mask.sum()} | Symfind: {sym_mask.sum()} | Both: {(rock_mask & sym_mask).sum()}")
display(selected.head(15))

## 3. Check the definitions

The first table compares both finders; the second compares the historical definition with association **at** the pre-infall snapshot. The latter is an alternative selection, not the default.

In [ ]:
from lmc_selection import membership_mask

at_pre = membership_mask(
    c.rockstar, c.rockstar, c.lmc_idx, i_pre, mode="at_pre_infall"
)
comparison = pd.DataFrame({
    "definition": ["Rockstar, ever before infall", "Symfind, ever before infall",
                   "Rockstar, at pre-infall", "Rockstar/Symfind intersection"],
    "N": [rock_mask.sum(), sym_mask.sum(), at_pre.sum(), (rock_mask & sym_mask).sum()],
})
display(comparison)

## 4. Run all Milky Way-est hosts

Write the full set of finder-specific index flags and a host summary. No additional completeness or radial cuts are added.

In [ ]:
members, summary = [], []
for i_host in sorted(LMC_INDEX_LEGACY):
    c = load_catalogs(BASE_MWEST, SUITE, i_host)
    i_pre, rock_mask, sym_mask = classify(c)
    members.append(membership_table(c, i_pre, rock_mask, sym_mask))
    summary.append({
        "host_index": i_host, "host_name": c.host_name, "lmc_idx": c.lmc_idx,
        "i_pre": i_pre, "a_pre": c.scale[i_pre],
        "N_rockstar": int(rock_mask.sum()),
        "N_symfind": int(sym_mask.sum()),
        "N_both": int((rock_mask & sym_mask).sum()),
    })

membership = pd.concat(members, ignore_index=True)
host_summary = pd.DataFrame(summary)
membership.to_csv(OUTDIR / "mwest_lmc_membership.csv", index=False)
host_summary.to_csv(OUTDIR / "mwest_lmc_host_summary.csv", index=False)
display(host_summary)

## 5. Diagnostic

Check per-host membership counts before distributing the catalog. Differences can arise from the finder tracking flags and positions.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(len(host_summary))
ax.plot(x, host_summary.N_rockstar, "o-", label="Rockstar", ms=4)
ax.plot(x, host_summary.N_symfind, "s-", label="Symfind", ms=4)
ax.set_xticks(x, host_summary.host_name, rotation=60, ha="right")
ax.set_ylabel("Historically LMC-associated branches")
ax.set_xlabel("MWest host")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 6. Data export and the raw-HLIST distinction

For the full time series, run `export_lmc_branches.py` from the same directory as this notebook. It saves all SymphonyLMC main branches, and the union of MWest LMC-associated branches, in separate compressed HDF5 files per host. The `mwest_lmc_membership.csv` index records the finder-specific classifications.

```bash
python export_lmc_branches.py --out exports
```

**Separate raw-HLIST analysis:** The corrected OJA figures read `lmc_associated_catalog_raw/Halo*_associated_nodes*.csv.gz` and propagate membership through `halo_id` → `desc_id` over raw snapshots. That is not the same set of IDs or necessarily the same population as this symlib selection. The notebook/script that generates those corrected raw catalogs is needed to reproduce that pipeline from the original HLIST files; it is not among the files included here.

## Appendix: corrected raw-HLIST catalog

For the corrected Rockstar **raw-HLIST** selection used in the later OJA figures, load the original `Halo*_associated_nodes*.csv.gz`. Association is propagated from progenitor `halo_id` to descendant `desc_id`; mass peaks are formed using only snapshots up to the pre-infall epoch. This calculation applies to nodes actually present in the input CSV, not automatically to every halo in the full HLIST tree.

In [ ]:
from lmc_selection import raw_hlist_preinfall_selection

RAW_DIR = Path("lmc_associated_catalog_raw")
RAW_HOST_INDEX = 0
RAW_HOST = "Halo004"
RAW_FILES = [
    RAW_DIR / f"{RAW_HOST}_associated_nodes_with_mpeak.csv.gz",
    RAW_DIR / f"{RAW_HOST}_associated_nodes.csv.gz",
    RAW_DIR / f"{RAW_HOST}_associated_nodes_phase_space.csv.gz",
]
RAW_CATALOG = next((p for p in RAW_FILES if p.exists()), None)

if RAW_CATALOG is not None:
    raw = pd.read_csv(RAW_CATALOG, dtype={"halo_id": "string", "desc_id": "string"})
    if "pre_or_at_infall" in raw:
        flags = raw["pre_or_at_infall"].astype(str).str.lower().isin(["true", "1", "t"])
        pre_rows = raw.loc[flags, ["raw_snapshot", "a"]].dropna()
        if pre_rows.empty:
            raise ValueError("No pre-infall epoch flagged in the raw catalog")
        raw_pre_snap = int(pre_rows.sort_values("a")["raw_snapshot"].iloc[-1])
    else:
        c_raw = load_catalogs(BASE_MWEST, SUITE, RAW_HOST_INDEX)
        if c_raw.host_name != RAW_HOST:
            raise ValueError("RAW_HOST_INDEX and RAW_HOST do not refer to the same host")
        symlib_pre, _, _ = classify(c_raw)
        a_pre = c_raw.scale[symlib_pre]
        raw_epochs = raw[["raw_snapshot", "a"]].dropna().drop_duplicates("raw_snapshot")
        if raw_epochs.empty:
            raise ValueError("Raw catalog has no snapshot epochs")
        j = np.abs(raw_epochs["a"].astype(float).to_numpy() - a_pre).argmin()
        raw_pre_snap = int(raw_epochs.iloc[j]["raw_snapshot"])

    at_infall = raw_hlist_preinfall_selection(raw, raw_pre_snap)
    raw_members = at_infall.loc[at_infall.causal_assoc]
    print(f"{RAW_HOST}: raw snapshot {raw_pre_snap}, associated nodes = {len(raw_members)}")
    display(raw_members[["halo_id", "desc_id", "Mvir", "Mpeak_upto_infall"]].head(10))
else:
    print(f"Optional corrected raw-HLIST catalog not found for {RAW_HOST} in {RAW_DIR}")